# 🔥 산불 모델 이어학습 (Colab T4)

Kaggle GPU 할당량 소진 시 Colab 무료 T4로 **남은 모델(internimage fold2 등)만** 이어학습.

**실행 전 필수:** 상단 메뉴 `런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU` 선택

원리: `FORCE_RETRAIN=set()` → 기존 가중치는 평가만, 없는 것만 학습.
T4(sm_75)는 기본 PyTorch 호환이라 P100 같은 재설치 불필요.

In [ ]:
# Cell 1: GPU 확인 + 코드 클론
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — 런타임을 T4로 변경하세요!')
assert torch.cuda.is_available(), 'T4 GPU 런타임으로 변경 후 다시 실행'

!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
%cd /content/fireimage_detection

In [ ]:
# Cell 2: 추가 패키지 (커스텀 모델용)
!pip install timm einops transformers -q
print('패키지 설치 완료')

In [ ]:
# Cell 3: Kaggle 인증 (새 토큰 방식 KGAT_...)
#   실행 후 입력창에 KGAT_... 붙여넣고 Enter
import os, getpass, re
!pip install kaggle -q
raw = getpass.getpass('Kaggle API Token (KGAT_...): ')
token = re.sub(r'[^A-Za-z0-9_\-]', '', raw)   # ASCII 토큰 문자만 (한글/공백 제거)
print('토큰 길이:', len(token), '| 시작:', token[:5], '(37, KGAT_ 이면 정상)')
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
# 인증 확인 (데이터셋 목록 뜨면 성공)
!kaggle datasets list --user yuntarwon

In [ ]:
# Cell 4: 데이터 다운로드 (Kaggle 데이터셋 3개 → data/fireimage/)
import os, glob, zipfile
BASE = '/content/fireimage_detection/data/fireimage'

def dl(dataset, dst):
    os.makedirs(dst, exist_ok=True)
    os.system(f'kaggle datasets download {dataset} -p {dst} --unzip')
    for _ in range(2):  # 중첩 zip 해제
        zips = glob.glob(f'{dst}/**/*.zip', recursive=True)
        if not zips:
            break
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(dst)
                os.remove(z)
            except Exception as e:
                print('unzip 실패', z, e)

dl('yuntarwon/fireimage-abnormal',         f'{BASE}/abnormal')
dl('yuntarwon/fireimage-abnormal-youtube', f'{BASE}/abnormal/youtube')
dl('yuntarwon/fireimage-normal',           f'{BASE}/normal')

imgs = ('.jpg', '.jpeg', '.png', '.bmp')
n = sum(1 for f in glob.glob(f'{BASE}/normal/**/*', recursive=True) if f.lower().endswith(imgs))
a = sum(1 for f in glob.glob(f'{BASE}/abnormal/**/*', recursive=True) if f.lower().endswith(imgs))
print(f'\nnormal: {n:,}장 / abnormal: {a:,}장')
assert n > 0 and a > 0, '데이터 다운로드 실패'

In [ ]:
# Cell 5: 이전 학습 가중치 복원 (resume — 완료된 20개 스킵용)
import glob, os, shutil
os.makedirs('/content/fireimage_detection/model_save', exist_ok=True)
os.system('kaggle kernels output yuntarwon/fireimage-training -p /content/prev')
n = 0
for pt in glob.glob('/content/prev/**/model_save/**/*.pt', recursive=True):
    i = pt.find('model_save/')
    rel = pt[i + len('model_save/'):]
    d = f'/content/fireimage_detection/model_save/{rel}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.copy2(pt, d)
    n += 1
print(f'복원된 가중치: {n}개 (없으면 0 → 전체 신규학습)')

In [ ]:
# Cell 6: 이어학습 실행 (기존 평가 + 없는 모델만 학습)
%cd /content/fireimage_detection
!python main_v2.py --class_name fireimage

In [ ]:
# Cell 7: 결과 확인
import pandas as pd
df = pd.read_csv('/content/fireimage_detection/results/fireimage/metrics.csv')
print(df.to_string())

In [ ]:
# Cell 8: 가중치·결과 다운로드 (세션 종료 전 반드시 실행)
import shutil
shutil.make_archive('/content/model_save_backup', 'zip', '/content/fireimage_detection', 'model_save')
shutil.make_archive('/content/results_backup', 'zip', '/content/fireimage_detection', 'results')
from google.colab import files
files.download('/content/model_save_backup.zip')
files.download('/content/results_backup.zip')